# Dart recognition pipeline

Stages: **1** load image, **2** detect the board, **3** rectify, **4** locate darts, **5** read numbers, **6** locate dart tips, **7** score, **8** combined view.

Stages 1-3 run the board detector. Stage 4 now uses an experimental **classical single-dart pipeline**, combining flight contrast with paired shaft edges. It uses only the core NumPy/OpenCV dependencies: no model weights or downloads. It may return no box when evidence is weak or ambiguous.

The optional pretrained OWLv2 adapter remains available in `dartcv.dart_detection` for comparison. Number reading and tip localisation remain unconfigured. A whole-dart box is not a tip, so boxes alone never produce scores.

All observations use **original-image pixels** `(x, y)`. Diagnostic views use a smaller working image. Only one input photograph is processed at a time.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

from dartcv.detect import DetectionError, detect_board
from dartcv.diagnostics import detection_overlays
from dartcv.geometry import compute_homography, score_from_image_point
from dartcv.overlay import draw_overlay, topdown_overlay


def show(img_bgr, title, width=7):
    plt.figure(figsize=(width, width * img_bgr.shape[0] / img_bgr.shape[1]))
    plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.axis('off')
    plt.show()

## Configuration

In [ ]:
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').exists())

IMAGE_PATH = REPO_ROOT / 'data/images/single_dart/PXL_20261018_152634348.jpg'
UPRIGHT_DEG = 0.0  # clockwise angle at which the 20 appears; resolves the 36-degree ambiguity
SHOW_DIAGNOSTICS = False  # also show every board-detection stage
MANUAL_CALIBRATION = None  # four (x, y) pixels for 20|1, 6|10, 3|19, 11|14 instead of auto-detection
MANUAL_TIPS = []  # optional hand-picked (x, y) tip pixels, scored with provenance 'manual'
EXPORT_PATH = None  # e.g. REPO_ROOT / 'out/recognition.png'; nothing is written when None

## 1. Load the image

In [ ]:
image = cv2.imread(str(IMAGE_PATH))
if image is None:
    raise FileNotFoundError(f'cannot read {IMAGE_PATH}')
image.flags.writeable = False  # later stages must draw on copies

print(IMAGE_PATH.name, 'height x width x channels:', image.shape)
show(image, f'1. Original: {IMAGE_PATH.name}')

## 2. Detect the board

The homography `H` maps board millimetres to image pixels. Rings follow the measured paint edges where available. If the 20 is not near the top of the image, set `UPRIGHT_DEG`; if detection fails, set `MANUAL_CALIBRATION`.

In [ ]:
if MANUAL_CALIBRATION is not None:
    detection = None
    H = compute_homography(MANUAL_CALIBRATION)
    print('calibration: manual')
else:
    try:
        detection = detect_board(image, upright_deg=UPRIGHT_DEG, diagnostics=SHOW_DIAGNOSTICS)
    except DetectionError as exc:
        raise RuntimeError(f'board detection failed ({exc}); set MANUAL_CALIBRATION') from exc
    H = detection.H
    bull = 'found' if detection.bull is not None else 'not found'
    print(f'calibration: automatic (ECC score {detection.ecc_score}, bull {bull}, ring edges {len(detection.rings)}/4)')

ring_contours = {} if detection is None else {name: ring.contour for name, ring in detection.rings.items()}
board_overlay = draw_overlay(
    image, H, numbers=False, ring_contours=ring_contours, thickness=max(1, max(image.shape) // 1600)
)
show(board_overlay, '2. Board geometry')

if SHOW_DIAGNOSTICS and detection is not None:
    for name, view in detection_overlays(image, detection).items():
        show(view, name)

## 3. Rectify

In [ ]:
show(topdown_overlay(image, H, numbers=False, ring_contours=ring_contours), '3. Top-down view (20 at the top)')

## Result contracts and backend slots

Each recognition stage has a backend slot that stays `None` until a model is connected. A stage reports `not configured` (no backend), `failed` (backend raised or returned invalid data) or `ok`; `ok` with zero items is a genuine empty result.

- `DART_DETECTOR(image) -> list[DartBox]`
- `NUMBER_READER(image, H) -> list[NumberReading]`
- `TIP_LOCATOR(image, boxes) -> list[DartTip]`

Backends must return original-image pixels; undo any crop or resize before returning. A dart box is not a tip: only a `DartTip` is scored.

In [ ]:
import numpy as np

from dartcv.classical_darts import ClassicalDartDetector
from dartcv.recognition import (
    DartBox,
    DartTip,
    NumberReading,
    StageResult,
    expected_sector,
    run_stage,
    validate_boxes,
    validate_numbers,
    validate_tips,
)
from dartcv.recognition_overlay import annotate

ENABLE_DART_DETECTION = True
CLASSICAL_FLIGHT_HSV = None
CLASSICAL_WORK_SIZE = 1200
CLASSICAL_MIN_SCORE = 0.65
CLASSICAL_AMBIGUITY_MARGIN = 0.05
SHOW_CLASSICAL_DIAGNOSTICS = True

_detector_options = {
    'H': H,
    'flight_hsv': CLASSICAL_FLIGHT_HSV,
    'work_size': CLASSICAL_WORK_SIZE,
    'min_score': CLASSICAL_MIN_SCORE,
    'ambiguity_margin': CLASSICAL_AMBIGUITY_MARGIN,
}
_cached_detector = globals().get('_session_dart_detector', globals().get('DART_DETECTOR'))
if ENABLE_DART_DETECTION:
    if not isinstance(_cached_detector, ClassicalDartDetector) or any(
        not np.array_equal(getattr(_cached_detector, name), value)
        for name, value in _detector_options.items()
    ):
        _cached_detector = ClassicalDartDetector(**_detector_options)
    _session_dart_detector = _cached_detector
    DART_DETECTOR = _session_dart_detector
else:
    DART_DETECTOR = None
NUMBER_READER = None
TIP_LOCATOR = None


def show_stage(title, result, base, **annotations):
    print(f'{title}: {result.message}')
    if result.status == 'ok':
        show(annotate(base, **annotations), title)

## 4. Locate whole darts

OWLv2 is an **open-vocabulary detector**: the prompt describes what to look for, and pretrained visual/text features produce candidate rectangles and match scores. Unlike a fixed-class detector, it can attempt a dart query without dart-specific training. This does not guarantee it recognises darts reliably.

The adapter converts BGR to RGB, lets the model processor resize/pad the image, and restores boxes to original pixels. It uses the original perspective, not the top-down image: the board homography does not describe protruding darts. Board calibration is not an input to this detector.

`DART_THRESHOLD` filters weak candidates; lowering it may recover missed darts but admits more false positives. Scores are not calibrated probabilities. `DART_NMS_IOU` controls duplicate suppression based on box overlap; overly aggressive suppression can also remove real overlapping darts. No fixed dart count is assumed.

Inspect whether each box encloses the flight **and** shaft/barrel, and whether wires, shadows, or the board itself were mistaken for darts. An empty result remains empty. One photograph is a feasibility check, not an accuracy measurement.

In [ ]:
darts = run_stage('dart detection', DART_DETECTOR, lambda items: validate_boxes(image, items), image)
show_stage('4. Dart locations', darts, image, boxes=darts.items)

if isinstance(DART_DETECTOR, ClassicalDartDetector) and DART_DETECTOR.diagnostics is not None:
    print(DART_DETECTOR.diagnostics.message)
    if SHOW_CLASSICAL_DIAGNOSTICS:
        for name, view in DART_DETECTOR.diagnostic_views().items():
            show(view, f'4. Classical: {name}')

## 5. Read the numbers

`NUMBER_READER` must return a box (`polygon`) and the recognised `text` for each printed number; the boxes are drawn on the full image, not cropped out. No number detector is configured yet, so this stage reports `not configured`. Readings are only compared with the model's sector order; a mismatch is reported, never overwritten. A systematic mismatch usually means the board orientation is wrong (see `UPRIGHT_DEG`).

In [ ]:
numbers = run_stage('number reading', NUMBER_READER, lambda items: validate_numbers(image, items), image, H)
show_stage('5. Number readings', numbers, image, numbers=numbers.items)

for r in numbers.items:
    expected = expected_sector(H, np.asarray(r.polygon, float).mean(axis=0))
    verdict = 'matches' if r.text == str(expected) else 'DIFFERS from'
    print(f'read {r.text!r} ({r.confidence:.2f}) {verdict} the model sector {expected}')

## 6. Locate dart tips

Tips come from `TIP_LOCATOR` or from `MANUAL_TIPS`; a box centre is never used as a tip.

In [ ]:
tip_stage = run_stage('tip localisation', TIP_LOCATOR, lambda items: validate_tips(image, items, darts.items), image, darts.items)
print(tip_stage.message)

manual_tips = tuple(DartTip(None, (float(x), float(y)), 1.0, 'manual') for x, y in MANUAL_TIPS)
validate_tips(image, manual_tips, ())
tips = tip_stage.items + manual_tips
print(f'{len(tips)} tip(s): {len(tip_stage.items)} detected, {len(manual_tips)} manual')

if tips:
    show(annotate(image, boxes=darts.items, tips=tips), '6. Dart tips')

## 7. Score

Scores use the standard board radii, not the fitted paint contours, and inherit the calibration's uncertainty. A score close to a ring or wire boundary is not reliable.

In [ ]:
scores = [score_from_image_point(H, t.xy) for t in tips]
if tips:
    print(f'{"dart":>5} {"tip (x, y)":>20} {"conf":>5} {"source":>8} {"score":>5}')
    for t, s in zip(tips, scores):
        dart = '-' if t.dart_id is None else t.dart_id
        print(f'{dart:>5} {t.xy[0]:>9.1f}, {t.xy[1]:>8.1f} {t.confidence:>5.2f} {t.provenance:>8} {s:>5}')
    print('total:', sum(scores))
else:
    print('No dart tips available: scores are unknown.')

## 8. Combined view

In [ ]:
combined = annotate(
    board_overlay, boxes=darts.items, numbers=numbers.items, tips=tips, scores=scores if tips else None
)
show(combined, '8. Board, darts, numbers, tips and scores')

if EXPORT_PATH is not None:
    EXPORT_PATH = Path(EXPORT_PATH)
    EXPORT_PATH.parent.mkdir(parents=True, exist_ok=True)
    if not cv2.imwrite(str(EXPORT_PATH), combined):
        raise OSError(f'could not write {EXPORT_PATH}')
    print('wrote', EXPORT_PATH)

## Connecting recognition models

`DART_DETECTOR` uses the optional local OWLv2 adapter. Change its settings in the contracts cell and re-run that cell and stage 4. Re-running the contracts cell reuses the same session detector and loaded weights when all settings match. Changing a setting constructs a new instance; disabling detection retains the cached detector for re-enabling. A kernel restart clears the session cache. Stage 4 still performs inference on every execution: this caches the loaded model, not recognition results.

Model ID and pinned revision come from `dartcv.dart_detection`, and each box records model/revision and matched prompt in its provenance. The session retains one detector configuration, not an unbounded history of models.

`NUMBER_READER` and `TIP_LOCATOR` remain unset. A later dart-specific detector can implement the same callable contract. Precise scoring still requires a measured tip, not a box centre. See [the pipeline documentation](../docs/cv-pipeline.md#pretrained-whole-dart-experiment) for preprocessing, limitations, and the training roadmap.